# Imports and Configs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchao
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import scipy
import keras
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
X = torch.randn((100, 3, 400))
y = torch.randint(0, 2, (100,))

ds = TensorDataset(X, y)
dl = DataLoader(ds, batch_size=5)

In [ ]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp_delme.p")
    print('Size (KB):', os.path.getsize("temp_delme.p")/1e3)
    os.remove('temp_delme.p')

# Pytorch Quantization

In [ ]:
class simpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv1d(3, 8, 3, bias=False),
                                   nn.BatchNorm1d(8),
                                   nn.ReLU(),
                                   nn.MaxPool1d(2, 2))
        self.conv2 = nn.Sequential(nn.Conv1d(8, 16, 3, dilation = 2, bias=False),
                                   nn.BatchNorm1d(16),
                                   nn.ReLU(),
                                   nn.MaxPool1d(2, 2))
        self.conv3 = nn.Sequential(nn.Conv1d(16, 32, 3, dilation = 2, bias=False),
                                   nn.BatchNorm1d(32),
                                   nn.ReLU(),
                                   nn.MaxPool1d(2, 2))
        self.flatten = nn.Flatten()

        x = torch.randn((1, 3, 400))
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.flatten(x)

        self.fc1 = nn.Linear(x.shape[1], 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = simpleCNN().to(device)

In [ ]:
model.state_dict()

odict_keys(['conv1.0.weight', 'conv1.1.weight', 'conv1.1.bias', 'conv1.1.running_mean', 'conv1.1.running_var', 'conv1.1.num_batches_tracked', 'conv2.0.weight', 'conv2.1.weight', 'conv2.1.bias', 'conv2.1.running_mean', 'conv2.1.running_var', 'conv2.1.num_batches_tracked', 'conv3.0.weight', 'conv3.1.weight', 'conv3.1.bias', 'conv3.1.running_mean', 'conv3.1.running_var', 'conv3.1.num_batches_tracked', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'])

In [ ]:
from torchsummary import summary
summary(model, (3, 400))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv1d-1               [-1, 8, 398]              72
       BatchNorm1d-2               [-1, 8, 398]              16
              ReLU-3               [-1, 8, 398]               0
         MaxPool1d-4               [-1, 8, 199]               0
            Conv1d-5              [-1, 16, 195]             384
       BatchNorm1d-6              [-1, 16, 195]              32
              ReLU-7              [-1, 16, 195]               0
         MaxPool1d-8               [-1, 16, 97]               0
            Conv1d-9               [-1, 32, 93]           1,536
      BatchNorm1d-10               [-1, 32, 93]              64
             ReLU-11               [-1, 32, 93]               0
        MaxPool1d-12               [-1, 32, 46]               0
          Flatten-13                 [-1, 1472]               0
           Linear-14                   

In [ ]:
class simpleQCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = torch.ao.quantization.QuantStub()
        self.conv1 = nn.Sequential(nn.Conv1d(3, 8, 3, bias=False),
                                   nn.BatchNorm1d(8),
                                   nn.ReLU(),
                                   nn.MaxPool1d(2, 2))
        self.conv2 = nn.Sequential(nn.Conv1d(8, 16, 3, dilation = 2, bias=False),
                                   nn.BatchNorm1d(16),
                                   nn.ReLU(),
                                   nn.MaxPool1d(2, 2))
        self.conv3 = nn.Sequential(nn.Conv1d(16, 32, 3, dilation = 2, bias=False),
                                   nn.BatchNorm1d(32),
                                   nn.ReLU(),
                                   nn.MaxPool1d(2, 2))
        self.flatten = nn.Flatten()

        x = torch.randn((1, 3, 400))
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.flatten(x)

        self.fc1 = nn.Linear(x.shape[1], 64)
        self.fc2 = nn.Linear(64, 2)
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x) # Quantize the input first
        x = x.contiguous()
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.dequant(x)
        return x

model = simpleCNN().to(device)
qmodel = simpleQCNN().to(device)

In [ ]:
qmodel.load_state_dict(model.state_dict())
qmodel.eval()

qmodel.qconfig = torch.ao.quantization.default_qconfig
torch.ao.quantization.prepare(qmodel, inplace=True)
qmodel

simpleQCNN(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (conv1): Sequential(
    (0): Conv1d(
      3, 8, kernel_size=(3,), stride=(1,), bias=False
      (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
    )
    (1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv1d(
      8, 16, kernel_size=(3,), stride=(1,), dilation=(2,), bias=False
      (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
    )
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv1d(
      16, 32, kernel_size=(3,), stride=(1,), dilation=(2,), bias=False
      (activation_post_proc

In [ ]:
pred = qmodel(torch.randn((5, 3, 400)).to(device))
qmodel

simpleQCNN(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=-3.9678664207458496, max_val=3.7866098880767822)
  )
  (conv1): Sequential(
    (0): Conv1d(
      3, 8, kernel_size=(3,), stride=(1,), bias=False
      (activation_post_process): MinMaxObserver(min_val=-2.7244369983673096, max_val=2.555699110031128)
    )
    (1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv1d(
      8, 16, kernel_size=(3,), stride=(1,), dilation=(2,), bias=False
      (activation_post_process): MinMaxObserver(min_val=-1.4035295248031616, max_val=1.3540544509887695)
    )
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv1d(
      16,

In [ ]:
torch.ao.quantization.convert(qmodel, inplace=True)
qmodel

simpleQCNN(
  (quant): Quantize(scale=tensor([0.0611], device='cuda:0'), zero_point=tensor([65], device='cuda:0'), dtype=torch.quint8)
  (conv1): Sequential(
    (0): QuantizedConv1d(3, 8, kernel_size=(3,), stride=(1,), scale=0.041575875133275986, zero_point=66, bias=False)
    (1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): QuantizedConv1d(8, 16, kernel_size=(3,), stride=(1,), scale=0.0217132605612278, zero_point=65, dilation=(2,), bias=False)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): QuantizedConv1d(16, 32, kernel_size=(3,), stride=(1,), scale=0.008907643146812916, zero_point=61, dilation=(2,), bias=False)
    (1): BatchNorm1d(32, eps=

In [ ]:
print(model.conv1[0].weight)
print(qmodel.conv1[0].weight())

Parameter containing:
tensor([[[-0.3151, -0.2587, -0.2211],
         [-0.0139,  0.3197,  0.1047],
         [ 0.1923,  0.1998, -0.0233]],

        [[ 0.0758,  0.1055,  0.0208],
         [ 0.2734, -0.2784,  0.3182],
         [-0.0280, -0.2355,  0.0085]],

        [[-0.0371, -0.0715,  0.2662],
         [-0.2945, -0.1571,  0.0483],
         [ 0.1269,  0.0141, -0.2483]],

        [[ 0.0587, -0.1044,  0.0024],
         [-0.2643, -0.0187, -0.1701],
         [ 0.0478, -0.2911, -0.2057]],

        [[ 0.2225, -0.2668,  0.0839],
         [ 0.2809,  0.0435,  0.1381],
         [-0.0396,  0.2611,  0.1955]],

        [[ 0.1374,  0.3149,  0.1824],
         [-0.2343, -0.0337,  0.1903],
         [-0.1889,  0.3031,  0.2258]],

        [[ 0.2151, -0.0153, -0.0707],
         [ 0.1858, -0.2776,  0.1009],
         [-0.2049, -0.2206, -0.2163]],

        [[ 0.1621, -0.2453, -0.0122],
         [-0.1371, -0.3083, -0.2368],
         [ 0.2284,  0.2202,  0.1776]]], device='cuda:0', requires_grad=True)
tensor([[[-0.

In [ ]:
print_size_of_model(model)
print_size_of_model(qmodel)

Size (KB): 393.762
Size (KB): 108.77


In [ ]:
qmodel(X.to(device))

IndexError: Dimension out of range (expected to be in range of [-3, 2], but got 3)

In [ ]:
F.mse_loss(model(X.to(device)), qmodel(X.to(device)))

IndexError: Dimension out of range (expected to be in range of [-3, 2], but got 3)

model param sizes

Size (KB): 393.762 fp32
Size (KB): 108.77 qint8

In [ ]:
qmodel.state_dict().keys()

odict_keys(['quant.scale', 'quant.zero_point', 'conv1.0.weight', 'conv1.0.bias', 'conv1.0.scale', 'conv1.0.zero_point', 'conv1.1.weight', 'conv1.1.bias', 'conv1.1.running_mean', 'conv1.1.running_var', 'conv1.1.num_batches_tracked', 'conv2.0.weight', 'conv2.0.bias', 'conv2.0.scale', 'conv2.0.zero_point', 'conv2.1.weight', 'conv2.1.bias', 'conv2.1.running_mean', 'conv2.1.running_var', 'conv2.1.num_batches_tracked', 'conv3.0.weight', 'conv3.0.bias', 'conv3.0.scale', 'conv3.0.zero_point', 'conv3.1.weight', 'conv3.1.bias', 'conv3.1.running_mean', 'conv3.1.running_var', 'conv3.1.num_batches_tracked', 'fc1.scale', 'fc1.zero_point', 'fc1._packed_params.dtype', 'fc1._packed_params._packed_params', 'fc2.scale', 'fc2.zero_point', 'fc2._packed_params.dtype', 'fc2._packed_params._packed_params'])

#Tf Quantization and TFLM Conversion

In [ ]:
import tensorflow as tf
import tensorflow_model_optimization.quantization as tfmotq
import numpy as np

print(f"Using TF version: {tf.__version__}")

ModuleNotFoundError: No module named 'tensorflow_model_optimization'

PQT:
https://www.tensorflow.org/model_optimization/guide/quantization/post_training

QAT example:
https://github.com/tensorflow/model-optimization/blob/master/tensorflow_model_optimization/g3doc/guide/quantization/training_example.ipynb

https://www.tensorflow.org/model_optimization/guide/quantization/training_example

https://medium.com/game-of-bits/optimizing-tensorflow-models-using-quantization-fb4d09b46fac

### Defining the Equivalent Keras Model

Here, we define a Keras model using the Functional API that has the same architecture as our PyTorch model. We also need to convert our PyTorch dummy data to NumPy arrays for use with TensorFlow.

In [ ]:
Xnp = X.numpy().transpose(0, 2, 1)
y_np = y.numpy()

def create_tf_model():
    inputs = tf.keras.Input(shape=(400, 3))
    x = tf.keras.layers.Conv1D(8, 3, use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling1D(2, 2)(x)

    x = tf.keras.layers.Conv1D(16, 3, dilation_rate=2, use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling1D(2, 2)(x)

    x = tf.keras.layers.Conv1D(32, 3, dilation_rate=2, use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling1D(2, 2)(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    outputs = tf.keras.layers.Dense(2)(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

tf_model = create_tf_model()
tf_model.summary()

In [ ]:
# Save the model to a file and check its size
keras_model_path = "float_model.h5"
tf_model.save(keras_model_path)

float_model_size_kb = os.path.getsize(keras_model_path) / 1e3
print(f"Size of Float32 Keras model (KB): {float_model_size_kb:.2f}")

### Converting and Quantizing to TensorFlow Lite

Now we use the `TFLiteConverter`. To enable full integer quantization, we need to provide a representative dataset. This serves the same purpose as the calibration data in PyTorch. The converter runs inference on this data to determine the quantization parameters.

In [ ]:
# 1. Create the TFLiteConverter
converter = tf.lite.TFLiteConverter.from_keras_model(tf_model)

# 2. Define the representative dataset generator
# This generator yields samples from our dataset for calibration
def representative_dataset_gen():
    for i in range(len(X_np)):
        # The model expects a batch of shape (1, 400, 3)
        yield [X_np[i:i+1].astype(np.float32)]

# 3. Configure the converter for quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
# Ensure the converter produces a fully integer-quantized model
# Some operations might not be supported and would fall back to float.
# This forces an error if any operation cannot be quantized.
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# 4. Convert the model
print("Converting and quantizing the model to TFLite...")
tflite_quant_model = converter.convert()
print("Conversion complete.")

# 5. Save the quantized TFLite model and check its size
tflite_model_path = "quant_model.tflite"
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_quant_model)

quant_model_size_kb = os.path.getsize(tflite_model_path) / 1e3
print(f"Size of Quantized Int8 TFLite model (KB): {quant_model_size_kb:.2f}")

### Verifying the TFLite Model

To use a `.tflite` model, you need the `tf.lite.Interpreter`. The process involves allocating tensors and invoking the interpreter. We'll run inference and compare the output to the original Keras model.

In [ ]:
# Get the predictions from the original float model
float_predictions = tf_model.predict(X_np)

# Run inference with the TFLite model
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

tflite_predictions = []
for x_val in X_np:
    # Check if the input type is quantized (int8)
    if input_details['dtype'] == np.int8:
        input_scale, input_zero_point = input_details["quantization"]
        x_val_quantized = (x_val / input_scale + input_zero_point).astype(input_details["dtype"])
    else:
        x_val_quantized = x_val.astype(input_details["dtype"])

    interpreter.set_tensor(input_details['index'], x_val_quantized[np.newaxis, ...])
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_details['index'])

    # De-quantize the output if needed
    if output_details['dtype'] == np.int8:
        output_scale, output_zero_point = output_details['quantization']
        output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale

    tflite_predictions.append(output_data[0])

tflite_predictions = np.array(tflite_predictions)

# Calculate the Mean Squared Error
mse = np.mean(np.square(float_predictions - tflite_predictions))
print(f"Mean Squared Error between original and TFLite model outputs: {mse:.6f}")